<a href="https://colab.research.google.com/github/Renyambar/Reny_MachineLearning/blob/main/JS03/JS03-Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Deskripsi Tugas
Pada tugas pratikum ini Anda akan menggunakan data "Wisconsin Breast Cancer". Data tersebut terdiri dari 569 data yang digunakan untuk mendiagnonis jenis kanker Malignant (M) dan Benign (B). Tugas Anda adalah,

1. Pisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.

2. Lakukan proses encoding pada kolom "diagnosis".

3. Lakukan proses standardisasi pada semua kolom yang memiliki nilai numerik.

4. Lakukan proses seleksi fitur. Anda dapat menggunakan SelectKBest.

5. Lakukan proses pengujian dengan model Logistic Regression seperti pada praktikum 1.

6. Anda dapat menggunakan model pipeline untuk mempermudah perkejaan Anda.

7. Berdasarkan hasil analisa Anda, berapa jumlah fitur terbaik yang dapat digunakan? Apa saja fitur tersebut?


##Langkah 1 - Import Library

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.metrics import accuracy_score, classification_report


##Langkah 2 - Load Data & Pengelompokan Variabel

In [5]:
# Load Data
df = pd.read_csv("wbc.csv")

# Buang variabel yang tidak dapat digunakan
df = df.drop(columns=["id", "Unnamed: 32"])

# Pisahkan target
y_raw = df["diagnosis"]
X = df.drop(columns=["diagnosis"])

# Semua kolom X adalah numerik
num_cols = X.columns.tolist()

## Langkah 3 - Encoding kolom diagnosis

In [6]:
le = LabelEncoder()
y = le.fit_transform(y_raw)   # B -> 0, M -> 1
print("Mapping label:", dict(zip(le.classes_, le.transform(le.classes_))))


Mapping label: {'B': np.int64(0), 'M': np.int64(1)}


## Langkah 4 - Standarisasi

In [7]:
# Data Numerik
num_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer([
    ("num", num_tf, num_cols),
])


## Langkah 5 - Seleksi Fitur dengan SelectKBest

In [8]:
from sklearn.feature_selection import f_classif
selector_filter = SelectKBest(score_func=f_classif, k=19)  # k awal, akan dicoba beberapa nilai di bawah

# Pipeline final
pipe_filter = Pipeline([
    ("prep", preprocess),      # ekstraksi fitur (imputasi + standardisasi)
    ("sel", selector_filter),  # seleksi fitur
    ("clf", LogisticRegression(max_iter=1000))  # Poin 5: uji dengan Logistic Regression
])


## Langkah 6 - Uji dengan Model Logistuc Regression

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

pipe_filter.fit(X_train, y_train)
pred = pipe_filter.predict(X_test)

print("=== Filter (ANOVA) + Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_))


=== Filter (ANOVA) + Logistic Regression ===
Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           B       0.97      1.00      0.99        72
           M       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Langkah 7 - Inspeksi Fitur Terpilih

In [10]:
# 1) Nama fitur setelah preprocess
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()
print("Nama fitur:", feat_names)
print()

# 2) Mask & skor fitur terpilih (SelectKBest)
sel = pipe_filter.named_steps["sel"]
mask = sel.get_support()
selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]
top = sorted(zip(selected_names, selected_scores), key=lambda t: t[1], reverse=True)
print(f"{len(selected_names)} fitur terpilih:")
for name, score in top:
    print(f"  {name:20s}  skor={score:.2f}")


Nama fitur: ['num__radius_mean' 'num__texture_mean' 'num__perimeter_mean'
 'num__area_mean' 'num__smoothness_mean' 'num__compactness_mean'
 'num__concavity_mean' 'num__concave points_mean' 'num__symmetry_mean'
 'num__fractal_dimension_mean' 'num__radius_se' 'num__texture_se'
 'num__perimeter_se' 'num__area_se' 'num__smoothness_se'
 'num__compactness_se' 'num__concavity_se' 'num__concave points_se'
 'num__symmetry_se' 'num__fractal_dimension_se' 'num__radius_worst'
 'num__texture_worst' 'num__perimeter_worst' 'num__area_worst'
 'num__smoothness_worst' 'num__compactness_worst' 'num__concavity_worst'
 'num__concave points_worst' 'num__symmetry_worst'
 'num__fractal_dimension_worst']

19 fitur terpilih:
  num__concave points_worst  skor=733.72
  num__perimeter_worst  skor=717.25
  num__radius_worst     skor=692.86
  num__concave points_mean  skor=684.53
  num__perimeter_mean   skor=548.41
  num__area_worst       skor=522.19
  num__radius_mean      skor=511.27
  num__area_mean        skor=4

Mencari Jumlah Fitur(k) Terbaik

In [11]:
hasil = []
for k in range(2, len(num_cols) + 1):
    pipe_k = Pipeline([
        ("prep", preprocess),
        ("sel", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    pipe_k.fit(X_train, y_train)
    pred_k = pipe_k.predict(X_test)
    acc = accuracy_score(y_test, pred_k)
    hasil.append((k, acc))

hasil_df = pd.DataFrame(hasil, columns=["k", "accuracy"]).sort_values("accuracy", ascending=False)
hasil_df.head(10)


,k,accuracy
12,14,0.982456
17,19,0.982456
18,20,0.982456
16,18,0.982456
10,12,0.973684
20,22,0.973684
19,21,0.973684
14,16,0.973684
15,17,0.973684
13,15,0.973684


In [12]:
best_k = int(hasil_df.iloc[0]["k"])
print(f"Jumlah fitur terbaik (k) = {best_k}, Accuracy = {hasil_df.iloc[0]['accuracy']:.4f}")


Jumlah fitur terbaik (k) = 14, Accuracy = 0.9825


Fit ulang pipeline final dengan k terbaik

In [13]:
final_pipe = Pipeline([
    ("prep", preprocess),
    ("sel", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=1000))
])
final_pipe.fit(X_train, y_train)
pred_final = final_pipe.predict(X_test)

print("Accuracy akhir:", accuracy_score(y_test, pred_final))
print(classification_report(y_test, pred_final, target_names=le.classes_))

feat_names = final_pipe.named_steps["prep"].get_feature_names_out()
mask = final_pipe.named_steps["sel"].get_support()
final_features = feat_names[mask]

print(f"\n{len(final_features)} fitur terbaik yang digunakan:")
for f in final_features:
    print(" -", f)


Accuracy akhir: 0.9824561403508771
              precision    recall  f1-score   support

           B       0.97      1.00      0.99        72
           M       1.00      0.95      0.98        42

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


14 fitur terbaik yang digunakan:
 - num__radius_mean
 - num__perimeter_mean
 - num__area_mean
 - num__compactness_mean
 - num__concavity_mean
 - num__concave points_mean
 - num__radius_se
 - num__perimeter_se
 - num__radius_worst
 - num__perimeter_worst
 - num__area_worst
 - num__compactness_worst
 - num__concavity_worst
 - num__concave points_worst


## Jawaban Poin 7

Berdasarkan hasil percobaan berbagai nilai *k*, akurasi tertinggi pada data test
(≈98%) dicapai pada **k = 19** (dan beberapa k di sekitarnya menghasilkan akurasi
yang sama/hampir sama). Fitur-fitur yang terpilih didominasi oleh kelompok `*_worst`
dan `*_mean` yang berkaitan dengan ukuran, bentuk, dan keteraturan sel (`radius`,
`perimeter`, `area`, `concavity`, `concave points`) — sejalan dengan pengetahuan medis
bahwa sel kanker ganas (Malignant) cenderung berukuran lebih besar dan bentuknya
lebih tidak beraturan dibanding sel jinak (Benign).